  ETL Pipeline — Hotel Booking Cancellation Analysis
**Corporate Intelligence Project | UFV Madrid**

---



## Library Installation

In [18]:
# Install openpyxl to export to excel
!pip install openpyxl -q
print('Librarie ready')

Librerías listas


## Upload csv File


In [19]:
from google.colab import files
import io

print('Select file')
uploaded = files.upload()

# Detect the uploaded file name
INPUT_FILE = list(uploaded.keys())[0]
print(f' File loaded: {INPUT_FILE}')

Seleccionar archivo


Saving hotel_bookings_cleaned.csv to hotel_bookings_cleaned (2).csv
 Archivo cargado: hotel_bookings_cleaned (2).csv


---
## STEP 1–2: Imports, Extracción y Análisis de Calidad

In [20]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

OUTPUT_FILE = 'hotel_bookings_cleaned.xlsx'


print('ETL PIPELINE - HOTEL BOOKING DEMAND DATASET')


# ── STEP 1: EXTRACT ────────────────────────────────────────────

df = pd.read_csv(INPUT_FILE)
print(f'  Rows loaded    : {df.shape[0]:,}')
print(f'  Columns loaded : {df.shape[1]}')
print(f'  Memory usage   : {df.memory_usage().sum()/1024**2:.2f} MB')

# ── STEP 2: EXPLORATORY DATA QUALITY ANALYSIS ──────────────────

missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print('  Missing Values Summary:')
for col in missing[missing > 0].index:
    print(f'    - {col}: {missing[col]:,} missing ({missing_pct[col]}%)')

dups = df.duplicated().sum()
print(f'  Duplicate rows detected : {dups}')
print(f'  ADR range: [{df["adr"].min():.2f}, {df["adr"].max():.2f}]')
print(f'  ADR negative values     : {(df["adr"] < 0).sum()}')

zero_guests = ((df['adults']==0) & (df['children'].fillna(0)==0)
               & (df['babies']==0)).sum()
print(f'  Zero-guest records      : {zero_guests}')
print(f'  Overall cancel rate     : {df["is_canceled"].mean()*100:.2f}%')

ETL PIPELINE - HOTEL BOOKING DEMAND DATASET
  Rows loaded    : 87,144
  Columns loaded : 44
  Memory usage   : 29.25 MB
  Missing Values Summary:
  Duplicate rows detected : 19
  ADR range: [0.00, 335.00]
  ADR negative values     : 0
  Zero-guest records      : 0
  Overall cancel rate     : 27.52%


## STEP 3: Data Cleaning

In [21]:

initial_rows = len(df)

# 3.1 Remove exact duplicates
df = df.drop_duplicates()
print(f'  After drop_duplicates              : {len(df):,} rows')

# 3.2 Handle missing values
df['company']  = df['company'].fillna('No Company')
df['agent']    = df['agent'].fillna(0).astype(int)
df['children'] = df['children'].fillna(0).astype(int)
top_country    = df['country'].mode()[0]
df['country']  = df['country'].fillna(top_country)
df['meal']     = df['meal'].replace('Undefined', 'SC')
print(f'  Missing country filled with mode   : {top_country}')

# 3.3 Remove invalid records
mask_no_guests = ((df['adults']==0) & (df['children']==0) & (df['babies']==0))
df = df[~mask_no_guests]
print(f'  After removing zero-guest records  : {len(df):,} rows')

df = df[df['adr'] >= 0]
print(f'  After removing negative ADR        : {len(df):,} rows')

adr_upper = df['adr'].quantile(0.999)
df = df[df['adr'] <= adr_upper]
print(f'  After removing ADR outliers (>{adr_upper:.2f}): {len(df):,}')

rows_removed = initial_rows - len(df)
print(f'  Total rows removed in cleaning     : {rows_removed:,} ({rows_removed/initial_rows*100:.2f}%)')

# 3.4 Data type corrections
df['reservation_status_date'] = pd.to_datetime(
    df['reservation_status_date'], errors='coerce')
df['is_canceled']       = df['is_canceled'].astype(int)
df['is_repeated_guest'] = df['is_repeated_guest'].astype(int)

  After drop_duplicates              : 87,125 rows
  Missing country filled with mode   : PRT
  After removing zero-guest records  : 87,125 rows
  After removing negative ADR        : 87,125 rows
  After removing ADR outliers (>314.65): 87,037
  Total rows removed in cleaning     : 107 (0.12%)


## STEP 4: Feature Engineering

In [22]:


df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df['estimated_revenue'] = df['adr'] * df['total_nights']
df['revenue_lost']      = df['estimated_revenue'] * df['is_canceled']

month_map = {'January':1,'February':2,'March':3,'April':4,
             'May':5,'June':6,'July':7,'August':8,
             'September':9,'October':10,'November':11,'December':12}
df['arrival_month_num'] = df['arrival_date_month'].map(month_map)
df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-'
    + df['arrival_month_num'].astype(str) + '-'
    + df['arrival_date_day_of_month'].astype(str), errors='coerce')

def get_season(m):
    if m in [12,1,2]:  return 'Winter'
    elif m in [3,4,5]: return 'Spring'
    elif m in [6,7,8]: return 'Summer'
    else:              return 'Autumn'
df['season'] = df['arrival_month_num'].apply(get_season)

def lead_cat(d):
    if d <= 7:    return '1-Short (<1 week)'
    elif d <= 30: return '2-Near (1-4 weeks)'
    elif d <= 90: return '3-Medium (1-3 months)'
    elif d <= 180:return '4-Long (3-6 months)'
    else:         return '5-Very Long (>6 months)'
df['lead_time_category'] = df['lead_time'].apply(lead_cat)

df['room_mismatch'] = (df['reserved_room_type'] != df['assigned_room_type']).astype(int)
df['has_children']  = ((df['children'] > 0) | (df['babies'] > 0)).astype(int)

# Composite Cancellation Risk Score (0-100)
df['risk_score'] = (
    (df['deposit_type'] == 'No Deposit').astype(int) * 35
    + (df['lead_time'] > 90).astype(int) * 25
    + (df['previous_cancellations'] > 0).astype(int) * 20
    + (df['distribution_channel'] == 'TA/TO').astype(int) * 10
    + (df['total_of_special_requests'] == 0).astype(int) * 10
)
df['risk_category'] = pd.cut(
    df['risk_score'], bins=[0,25,50,75,100],
    labels=['Low Risk','Medium Risk','High Risk','Very High Risk'],
    include_lowest=True
)

print('  Features engineered: total_nights, total_guests,')
print('    estimated_revenue, revenue_lost, arrival_date,')
print('    season, lead_time_category, room_mismatch,')
print('    has_children, risk_score, risk_category')

  Features engineered: total_nights, total_guests,
    estimated_revenue, revenue_lost, arrival_date,
    season, lead_time_category, room_mismatch,
    has_children, risk_score, risk_category


## STEP 5: Final Validation

In [23]:

print(f'  Final shape              : {df.shape}')
print(f'  Remaining nulls          : {df.isnull().sum().sum()}')
print(f'  Cancellation rate        : {df["is_canceled"].mean()*100:.2f}%')
print(f'  City Hotel records       : {(df["hotel"]=="City Hotel").sum():,}')
print(f'  Resort Hotel records     : {(df["hotel"]=="Resort Hotel").sum():,}')
print(f'  Date range               : {df["arrival_date"].min().date()} to {df["arrival_date"].max().date()}')
print(f'  Est. total revenue       : EUR {df["estimated_revenue"].sum():,.0f}')
print(f'  Revenue lost (cancel)    : EUR {df["revenue_lost"].sum():,.0f}')
print('  Risk Category Distribution:')
print(df['risk_category'].value_counts().sort_index().to_string())

  Final shape              : (87037, 44)
  Remaining nulls          : 0
  Cancellation rate        : 27.52%
  City Hotel records       : 53,245
  Resort Hotel records     : 33,792
  Date range               : 2015-07-01 to 2017-08-31
  Est. total revenue       : EUR 34,188,835
  Revenue lost (cancel)    : EUR 11,392,804
  Risk Category Distribution:
risk_category
Low Risk            236
Medium Risk       38079
High Risk         36484
Very High Risk    12238


## STEP 6: Load — Export to Excel and download

In [26]:
#Convert 'category' columns to string for Excel compatibility
for col in df.select_dtypes(include='category').columns: df[col] = df[col].astype(str)

#Export to Excel
df.to_excel(OUTPUT_FILE, index=False, engine='openpyxl')

file_size_kb = os.path.getsize(OUTPUT_FILE) / 1024
print(f' Saved to : {OUTPUT_FILE}')
print(f' File size : {file_size_kb:.1f} KB')

print('\n' + '=' * 60)
print('ETL PIPELINE COMPLETED')

# Automatic download in colab
from google.colab import files
files.download(OUTPUT_FILE)
print(f'Download Started: {OUTPUT_FILE}')

 Saved to : hotel_bookings_cleaned.xlsx
 File size : 14023.1 KB

ETL PIPELINE COMPLETED


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descarga iniciada: hotel_bookings_cleaned.xlsx
